# Lesson 13: Building a Mini Pipeline

## From Toy Agents to Real Pipelines

In Lesson 12, you chained 2 agents with simple text passing. In Module 4, you'll see the production pipeline with nested schemas, database tracking, and error handling.

This lesson **bridges the gap**. We build a 3-agent mini pipeline that introduces:
- **Nested schemas** — a list of objects inside another object (the key concept for Module 4)
- **Longer, more precise instructions** — closer to production quality
- **The sys.path.insert pattern** — how notebooks import production code

By the end, you'll be comfortable reading the production pipeline code.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../output/backend"))

from dotenv import load_dotenv
load_dotenv()

from agno.agent import Agent
from agno.models.anthropic import Claude
from tools.search import DataForSEOSearchTools
from tools.aio import get_dataforseo_credentials
from pydantic import BaseModel, Field

## Progressive Schema Building

In Lesson 11, you used a simple schema:

```python
class ArticleOutline(BaseModel):
    title: str
    sections: list[str]      # ← list of strings
    target_keyword: str
```

That works, but real outlines need **more detail per section**. Each section needs its own heading, key points, AND keywords.

That requires a **nested schema** — a model inside a model.

In [ ]:
# Step 1: What you already know (from Lesson 11)
class SimpleOutline(BaseModel):
    title: str = Field(description="Article title")
    sections: list[str] = Field(description="Section headings")

print("SimpleOutline — each section is just a string:")
print('  sections: ["Introduction", "Main Topic", "Conclusion"]')
print()

# Step 2: NEW — Define a Section as its own model
class Section(BaseModel):
    heading: str = Field(description="Section heading (H2)")
    key_points: list[str] = Field(description="Key points to cover in this section")
    keywords: list[str] = Field(default_factory=list, description="Target keywords for this section")

# Step 3: Use Section inside the outline
class DetailedOutline(BaseModel):
    title: str = Field(description="SEO-optimized article title")
    target_keyword: str = Field(description="Primary target keyword")
    sections: list[Section] = Field(description="Ordered list of article sections")

print("DetailedOutline — each section is a full object:")
print('  sections: [')
print('    Section(heading="Introduction", key_points=["What is SEO", "Why it matters"], keywords=["SEO"]),')
print('    Section(heading="On-Page Factors", key_points=["Title tags", "Meta descriptions"], keywords=["on-page SEO"]),')
print('  ]')
print()
print("The key difference: list[str] vs list[Section]")
print("  list[str]     = a list of plain text strings")
print("  list[Section] = a list where each item is a Section object with its own fields")

## Understanding Nested Schemas

Here's a visual comparison:

```
SimpleOutline                    DetailedOutline
├── title: str                   ├── title: str
├── sections: list[str]          ├── target_keyword: str
│   ├── "Introduction"           └── sections: list[Section]
│   ├── "Main Topic"                 ├── Section
│   └── "Conclusion"                 │   ├── heading: str
│                                    │   ├── key_points: list[str]
│                                    │   └── keywords: list[str]
│                                    ├── Section
│                                    │   ├── heading: ...
│                                    │   └── ...
│                                    └── ...
```

The production `ContentOutline` in `agents/schemas.py` uses this exact pattern — `OutlineSection` is the nested model, like our `Section` above.

## Build 3 Agents

Now we build a mini pipeline with 3 agents:
1. **Researcher** — searches the web, returns research notes (plain text)
2. **Outliner** — takes research, returns a `DetailedOutline` (nested schema)
3. **Writer** — takes the outline, writes a short article (plain Markdown)

Notice: the instructions are **longer and more precise** than in Lessons 8-12. Closer to production quality.

> **Cost:** ~$0.20-0.40 for all three agents (Claude Sonnet). Takes 1-2 minutes total.

In [ ]:
# Set up search tools (same conditional pattern as Lesson 9)
creds = get_dataforseo_credentials()
search_tools = []
if creds:
    search_tools = [DataForSEOSearchTools(login=creds[0], password=creds[1])]
    print("DataForSEO search enabled!")
else:
    print("No DataForSEO key — researcher will use built-in knowledge only.")

# Agent 1: Researcher — searches the web
researcher = Agent(
    name="Mini Researcher",
    model=Claude(id="claude-sonnet-4-5-20250929"),
    tools=search_tools,
    instructions=[
        "You are an expert SEO researcher.",
        "Research the given topic using web search.",
        "Find primary keywords, secondary keywords, and what competitors cover.",
        "Identify content gaps — what's missing from existing articles.",
        "Return organized research notes with clear sections.",
    ],
)

# Agent 2: Outliner — creates structured outline
outliner = Agent(
    name="Mini Outliner",
    model=Claude(id="claude-sonnet-4-5-20250929"),
    output_schema=DetailedOutline,
    instructions=[
        "You are an expert content strategist.",
        "Given research notes, create a structured article outline.",
        "Include 4-6 sections with clear H2 headings.",
        "Each section must have 2-4 key points and at least 1 target keyword.",
        "The title should be SEO-optimized and compelling.",
    ],
)

# Agent 3: Writer — writes from the outline
writer = Agent(
    name="Mini Writer",
    model=Claude(id="claude-sonnet-4-5-20250929"),
    instructions=[
        "You are an expert SEO content writer.",
        "Write a concise article (500-800 words) following the provided outline exactly.",
        "Use Markdown: H1 for title, H2 for sections.",
        "Cover every key point listed in each section.",
        "Naturally incorporate the target keywords.",
        "Write in a clear, professional tone.",
    ],
    markdown=True,
)

print("Created 3 agents: Researcher, Outliner, Writer")

In [ ]:
topic = "How to write SEO-friendly blog titles"

# Step 1: Research
print("Step 1: Researching...")
research = researcher.run(f"Research this topic for an SEO article: {topic}")
print(f"  Done! ({len(research.content)} chars of research notes)\n")

# Step 2: Outline (with nested schema!)
print("Step 2: Creating outline...")
outline_response = outliner.run(
    f"Create a detailed article outline from these research notes:\n\n{research.content}"
)
outline = outline_response.content
print(f"  Title: {outline.title}")
print(f"  Keyword: {outline.target_keyword}")
print(f"  Sections: {len(outline.sections)}\n")

# Step 3: Accessing nested data — this is the key skill!
print("Accessing nested data:")
for i, section in enumerate(outline.sections, 1):
    print(f"  Section {i}: {section.heading}")
    print(f"    Key points: {section.key_points}")
    print(f"    Keywords: {section.keywords}")
    print()

In [ ]:
# Step 3: Write the article from the outline
print("Step 3: Writing article...")

# Convert outline to string for the writer
outline_text = f"Title: {outline.title}\nKeyword: {outline.target_keyword}\n\n"
for section in outline.sections:
    outline_text += f"## {section.heading}\n"
    outline_text += f"Key points: {', '.join(section.key_points)}\n"
    outline_text += f"Keywords: {', '.join(section.keywords)}\n\n"

article = writer.run(f"Write an article from this outline:\n\n{outline_text}")
print(f"  Done! ({len(article.content.split())} words)\n")
print("--- Article Preview (first 500 chars) ---")
print(article.content[:500])
print("\n...")

## Reading Production Code

Compare our mini pipeline with the real one. The pattern is similar:

```python
# Mini pipeline (what we just built)          # Real product (content_writer.py)
research = researcher.run(topic)               # Content Writer does research + writing
outline = outliner.run(research.content)        # in a single agent — simpler!
article = writer.run(outline_text)              response = content_writer.run(topic)
```

The real product simplifies things further:
- **One agent** (Content Writer) handles both research and writing
- **Storage tools** — `save_article()` saves to disk automatically
- **Error handling** — `try/except` wraps each step
- **File output** — saves the article as a `.md` file in `content/`
- **Image enrichment** — optional Image Finder agent step

## Importing Production Code from Notebooks

The next cell uses `sys.path.insert(0, '../../output')`. Here's what this does:

When Python imports a module (like `from agents import research_agent`), it searches a list of folders called the **module search path**. By default, notebooks only look in their own folder.

Our production code lives in `output/` — two levels up from this notebook. `sys.path.insert(0, '../../output')` adds that folder to Python's search path, so imports like `from agents import ...` work.

You'll see this same pattern in Lessons 16-19 and 21 — any notebook that uses the production code needs it.

In [ ]:
import sys, os

# This tells Python: "also look in the output/ folder for imports"
# We need this because the notebooks are in lessons_en/, not output/
sys.path.insert(0, os.path.abspath("../../output/backend"))

# Now we can import production code!
from agents import ContentOutline, OutlineSection

print("Production ContentOutline schema:")
print(f"  Fields: {list(ContentOutline.model_fields.keys())}")
print()
print("Production OutlineSection schema:")
print(f"  Fields: {list(OutlineSection.model_fields.keys())}")
print()
print("Compare with our DetailedOutline:")
print(f"  Fields: {list(DetailedOutline.model_fields.keys())}")
print()
print("Compare with our Section:")
print(f"  Fields: {list(Section.model_fields.keys())}")
print()
print("Same pattern! The production version just has more fields.")

## Mini vs Production

| Feature | Mini Pipeline (this lesson) | Real Product |
|---------|---------------------------|---------------------------|
| Agents | 3 separate (researcher, outliner, writer) | 3 members: Content Writer, Image Finder, AIO Analyzer |
| Research + Write | Separate agents | Combined in Content Writer |
| Schema | `DetailedOutline` with `Section` | `ContentOutline` with `OutlineSection` (optional) |
| Model | Claude Sonnet for all | Claude Sonnet for all |
| Storage | None | Local file storage (JSON + .md) |
| Error handling | None | try/except at each step |
| Cost | ~$0.20-0.40 | ~$0.10-0.30 per article |

The **pattern** is the same: agent + model + tools + instructions. The production version is actually simpler (fewer agents) but adds storage and error handling.

## Lesson 13 Summary

What you learned:
- **Nested schemas** — `list[Section]` instead of `list[str]` for richer data
- **Accessing nested data** — `outline.sections[0].heading`, `outline.sections[0].key_points`
- **3-agent pipeline** — researcher → outliner → writer, each with focused instructions
- **`sys.path.insert`** — how notebooks import production code from `output/backend/`
- **Mini vs real product** — same pattern, production combines research + writing into one agent and adds storage

### Module 3 Summary — Building Agents

| Lesson | Concept |
|--------|---------|
| Lesson 8 | Create a basic agent with name, model, instructions |
| Lesson 9 | Add tools — agents can search the web |
| Lesson 11 | Structured output — agents return formatted data |
| Lesson 12 | Chain agents — output of one becomes input for next |
| Lesson 13 | Mini pipeline — nested schemas, 3-agent chain, production patterns |

**Next module:** Module 4 — Building the Product. You'll see the production agents, the team that coordinates them, and how to extend the system with Claude Code.

## Exercise

Modify the `DetailedOutline` schema:
1. Add a `tone` field (str, description "Writing tone for the article, e.g., 'professional', 'casual'")
2. Add a `target_audience` field (str, description "Who this article is written for")
3. Re-run the outliner with the updated schema
4. Verify the new fields are populated: `outline.tone`, `outline.target_audience`

This is how you'd customize the production schema — add fields, re-run, verify.

In [ ]:
# Step 1: Define the extended schema
class ExtendedOutline(___):
    title: str = Field(description="SEO-optimized article title")
    target_keyword: str = Field(description="Primary target keyword")
    sections: list[Section] = Field(description="Ordered list of article sections")
    tone: ___ = Field(description="Writing tone for the article, e.g., 'professional', 'casual'")
    target_audience: ___ = Field(description="Who this article is written for")

# Step 2: Create a new outliner with the updated schema
extended_outliner = Agent(
    name="Extended Outliner",
    model=Claude(id="claude-sonnet-4-5-20250929"),
    output_schema=___,
    instructions=[
        "You are an expert content strategist.",
        "Given research notes, create a structured article outline.",
        "Include 4-6 sections with clear H2 headings.",
        "Each section must have 2-4 key points and at least 1 target keyword.",
        "Also specify the writing tone and target audience.",
    ],
)

# Step 3: Run with existing research
extended_response = extended_outliner.run(
    f"Create a detailed article outline from these research notes:\n\n{research.___}"
)
extended_outline = extended_response.___

# Step 4: Verify new fields
print(f"Title: {extended_outline.title}")
print(f"Tone: {extended_outline.___}")
print(f"Target audience: {extended_outline.___}")

<details>
<summary>Click to reveal answer</summary>

```python
class ExtendedOutline(BaseModel):
    title: str = Field(description="SEO-optimized article title")
    target_keyword: str = Field(description="Primary target keyword")
    sections: list[Section] = Field(description="Ordered list of article sections")
    tone: str = Field(description="Writing tone for the article, e.g., 'professional', 'casual'")
    target_audience: str = Field(description="Who this article is written for")

extended_outliner = Agent(
    name="Extended Outliner",
    model=Claude(id="claude-sonnet-4-5-20250929"),
    output_schema=ExtendedOutline,
    instructions=[
        "You are an expert content strategist.",
        "Given research notes, create a structured article outline.",
        "Include 4-6 sections with clear H2 headings.",
        "Each section must have 2-4 key points and at least 1 target keyword.",
        "Also specify the writing tone and target audience.",
    ],
)

extended_response = extended_outliner.run(
    f"Create a detailed article outline from these research notes:\n\n{research.content}"
)
extended_outline = extended_response.content

print(f"Title: {extended_outline.title}")
print(f"Tone: {extended_outline.tone}")
print(f"Target audience: {extended_outline.target_audience}")
```
</details>

## Ready for Module 4? — Quick Check

Before moving on, make sure you understand these 3 key concepts. Try to answer each without looking back.

**1. What's the difference between `list[str]` and `list[Section]`?**

<details>
<summary>Click to reveal answer</summary>

`list[str]` is a list of plain text strings (e.g., `["Intro", "Main", "Conclusion"]`). `list[Section]` is a list where each item is a `Section` object with its own fields (heading, key_points, keywords). Nested schemas give you richer, more structured data.
</details>

**2. Why does the real product combine research and writing into one agent?**

<details>
<summary>Click to reveal answer</summary>

Claude Sonnet is capable enough to handle both research (web search) and writing in a single agent call. This is simpler (fewer files, fewer handoffs) and faster (one call instead of chaining 2-3 agents). The Content Writer uses tools for search and storage, so it can research, write, and save — all in one step.
</details>

**3. What does `sys.path.insert(0, '../../output/backend')` do?**

<details>
<summary>Click to reveal answer</summary>

It adds the `output/backend/` directory to Python's module search path. Without it, notebooks in `lessons_en/` can't find and import production code like `from agents import content_writer`. Every notebook that uses production code needs this line.
</details>